# Pandas Interview Preparation

Comprehensive, interview-focused reference covering Series/DataFrame creation, IO, indexing & selection, groupby/aggregation, reshaping, time series, performance tips, and common interview exercises with concise examples.

## Setup: import pandas and check versions

In [2]:
import pandas as pd
import numpy as np
print('pandas:', pd.__version__, 'numpy:', np.__version__)


pandas: 3.0.1 numpy: 2.4.3


In [3]:
import pandas as pd
import pyarrow as pa
print(pd.__version__)
print(pa.__version__)


3.0.1
23.0.1


## Series and DataFrame creation (common patterns)

In [4]:
s = pd.Series([10,20,30], index=['a','b','c'])
print(s)
df = pd.DataFrame({'A':[1,2,3], 'B':[4,5,6]})
print(df)
arr = np.random.randn(4,3)
df2 = pd.DataFrame(arr, columns=list('XYZ'))
print(df2.head())


a    10
b    20
c    30
dtype: int64
   A  B
0  1  4
1  2  5
2  3  6
          X         Y         Z
0  0.235557  0.994477 -0.250861
1  0.246938  1.990681  1.520521
2 -1.302635 -1.544015  1.025313
3 -1.567182 -0.356451  0.874121


## Reading / Writing data (CSV, Excel, Parquet) — concise examples

In [5]:
# read CSV (specify dtypes for performance)
df = pd.read_csv('data/data.csv', dtype={'name':str, 'age':int})
df
df.to_parquet('data/data.parquet')
print('I/O examples: pd.read_csv / df.to_parquet')


I/O examples: pd.read_csv / df.to_parquet


## Inspecting data: head, tail, info, describe

In [6]:
print(df2.head())
print(df2.info())
print(df2.describe())


          X         Y         Z
0  0.235557  0.994477 -0.250861
1  0.246938  1.990681  1.520521
2 -1.302635 -1.544015  1.025313
3 -1.567182 -0.356451  0.874121
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       4 non-null      float64
 1   Y       4 non-null      float64
 2   Z       4 non-null      float64
dtypes: float64(3)
memory usage: 228.0 bytes
None
              X         Y         Z
count  4.000000  4.000000  4.000000
mean  -0.596830  0.271173  0.792273
std    0.973748  1.545822  0.748216
min   -1.567182 -1.544015 -0.250861
25%   -1.368771 -0.653342  0.592875
50%   -0.533539  0.319013  0.949717
75%    0.238402  1.243528  1.149115
max    0.246938  1.990681  1.520521


## Indexing and selection: `loc`, `iloc`, boolean indexing, `at`/`iat`

In [7]:
df = pd.DataFrame({'name':['a','b','c','d'], 'score':[9,8,7,10], 'group':['x','y','x','y']})
print(df.loc[0])
print(df.iloc[0:2])
print(df[df['score']>8])
df.at[0,'score'] = 11  # fast scalar set
print(df)


name     a
score    9
group    x
Name: 0, dtype: object
  name  score group
0    a      9     x
1    b      8     y
  name  score group
0    a      9     x
3    d     10     y
  name  score group
0    a     11     x
1    b      8     y
2    c      7     x
3    d     10     y


## Adding / transforming columns: vectorized ops, `assign`, `apply`

In [8]:
df['score2'] = df['score'] * 2  # vectorized
print(df)
df = df.assign(score_norm = lambda d: (d.score - d.score.mean()) / d.score.std())
print(df)
# use apply for row-wise logic (slower); prefer vectorized ops when possible
df['top'] = df['score'].apply(lambda x: x>8)
print(df)


  name  score group  score2
0    a     11     x      22
1    b      8     y      16
2    c      7     x      14
3    d     10     y      20
  name  score group  score2  score_norm
0    a     11     x      22    1.095445
1    b      8     y      16   -0.547723
2    c      7     x      14   -1.095445
3    d     10     y      20    0.547723
  name  score group  score2  score_norm    top
0    a     11     x      22    1.095445   True
1    b      8     y      16   -0.547723  False
2    c      7     x      14   -1.095445  False
3    d     10     y      20    0.547723   True


## GroupBy: aggregation, transform, filter, iteration

In [9]:
g = df.groupby('group')
print(g['score'].mean())
# aggregate multiple functions
print(g.agg({'score':['mean','max'], 'score2':'sum'}))
# transform preserves index (useful for adding columns)
df['group_mean'] = g['score'].transform('mean')
print(df)


group
x    9.0
y    9.0
Name: score, dtype: float64
      score     score2
       mean max    sum
group                 
x       9.0  11     36
y       9.0  10     36
  name  score group  score2  score_norm    top  group_mean
0    a     11     x      22    1.095445   True         9.0
1    b      8     y      16   -0.547723  False         9.0
2    c      7     x      14   -1.095445  False         9.0
3    d     10     y      20    0.547723   True         9.0


## Merging, joining and concatenation

In [10]:
left = pd.DataFrame({'id':[1,2,3], 'val':['a','b','c']})
right = pd.DataFrame({'id':[2,3,4], 'val2':[10,20,30]})
print(pd.merge(left, right, on='id', how='inner'))
print(pd.concat([left, right], ignore_index=True, sort=False))


   id val  val2
0   2   b    10
1   3   c    20
   id  val  val2
0   1    a   NaN
1   2    b   NaN
2   3    c   NaN
3   2  NaN  10.0
4   3  NaN  20.0
5   4  NaN  30.0


## Reshaping: pivot, pivot_table, melt, stack/unstack, wide<->long

In [17]:
df_long = pd.DataFrame({'name':['City','City','Elevate','Elevate'], 'year':[2020,2021,2020,2021], 'val':[1,2,3,4]})
print(df_long)
print('#'*30)
print(df_long.pivot(index='name', columns='year', values='val'))
print('#'*30)
print(pd.pivot_table(df_long, index='name', columns='year', values='val', aggfunc='sum'))
print('#'*30)
print(df_long.melt(id_vars=['name'], value_vars=['val']))


      name  year  val
0     City  2020    1
1     City  2021    2
2  Elevate  2020    3
3  Elevate  2021    4
##############################
year     2020  2021
name               
City        1     2
Elevate     3     4
##############################
year     2020  2021
name               
City        1     2
Elevate     3     4
##############################
      name variable  value
0     City      val      1
1     City      val      2
2  Elevate      val      3
3  Elevate      val      4


## Missing data: detect, drop, fill, interpolate

In [21]:
dfm = pd.DataFrame({'A':[1,np.nan,3,np.nan], 'B':[np.nan,2,3,np.nan]})
print(dfm)
print('#'*30)
print(dfm.isna())
print('#'*30)
print(dfm.dropna())
print('#'*30)
print(dfm.fillna(0))
print('#'*30)
print(dfm.interpolate())


     A    B
0  1.0  NaN
1  NaN  2.0
2  3.0  3.0
3  NaN  NaN
##############################
       A      B
0  False   True
1   True  False
2  False  False
3   True   True
##############################
     A    B
2  3.0  3.0
##############################
     A    B
0  1.0  0.0
1  0.0  2.0
2  3.0  3.0
3  0.0  0.0
##############################
     A    B
0  1.0  NaN
1  2.0  2.0
2  3.0  3.0
3  3.0  3.0


## Datetime: to_datetime, indexing, resample, shifting

In [25]:
d = pd.DataFrame({'date':['2020-01-01','2020-01-02','2020-01-05'], 'value':[1,2,3]})
print(d)
print('#'*30)
d['date'] = pd.to_datetime(d['date'])
d = d.set_index('date')
print(d.resample('D').asfreq())
print('#'*30)
print(d.resample('D').ffill())
print('#'*30)
print(d.shift(1))  # lag by 1


         date  value
0  2020-01-01      1
1  2020-01-02      2
2  2020-01-05      3
##############################
            value
date             
2020-01-01    1.0
2020-01-02    2.0
2020-01-03    NaN
2020-01-04    NaN
2020-01-05    3.0
##############################
            value
date             
2020-01-01      1
2020-01-02      2
2020-01-03      2
2020-01-04      2
2020-01-05      3
##############################
            value
date             
2020-01-01    NaN
2020-01-02    1.0
2020-01-05    2.0


## Categorical dtype, memory and performance tips

In [26]:
df_cat = pd.DataFrame({'city': ['ny','la','ny','sf','la']})
print(df_cat['city'].astype('category').cat.categories)
# use `dtype` in read_csv for large data to save memory


Index(['la', 'ny', 'sf'], dtype='str')


## Window functions: rolling, expanding, ewm

In [27]:
ts = pd.Series([1,2,3,4,5])
print(ts.rolling(3).mean())
print(ts.expanding().sum())
print(ts.ewm(alpha=0.5).mean())


0    NaN
1    NaN
2    2.0
3    3.0
4    4.0
dtype: float64
0     1.0
1     3.0
2     6.0
3    10.0
4    15.0
dtype: float64
0    1.000000
1    1.666667
2    2.428571
3    3.266667
4    4.161290
dtype: float64


## MultiIndex and advanced indexing

In [28]:
arrays = [[1,1,2,2],[ 'a','b','a','b']]
mi = pd.MultiIndex.from_arrays(arrays, names=('num','let'))
dfm = pd.DataFrame({'val':[10,20,30,40]}, index=mi)
print(dfm.index.names)
print(dfm.loc[(1,'a')])


['num', 'let']
val    10
Name: (1, a), dtype: int64


## Useful pandas functions & helpers (examples)

In [30]:
# value_counts / normalize / sort_values
s = pd.Series(['a','b','a','c','b','a'])
print('value_counts:', s.value_counts())
print('proportions:', s.value_counts(normalize=True))
# nunique / unique / factorize
print('nunique:', s.nunique(), 'unique:', s.unique())
print('factorize:', pd.factorize(s))
# map / replace / applymap (elementwise on DataFrame)
dfx = pd.DataFrame({'col':['a','b','c'], 'v':[1,2,3]})
print(dfx.replace({'a':'A'}))
print(dfx['col'].map({'a':'A','b':'B'}))
print(dfx.map(lambda x: str(x).upper()))


value_counts: a    3
b    2
c    1
Name: count, dtype: int64
proportions: a    0.500000
b    0.333333
c    0.166667
Name: proportion, dtype: float64
nunique: 3 unique: <ArrowStringArray>
['a', 'b', 'c']
Length: 3, dtype: str
factorize: (array([0, 1, 0, 2, 1, 0]), Index(['a', 'b', 'c'], dtype='str'))
  col  v
0   A  1
1   b  2
2   c  3
0      A
1      B
2    NaN
Name: col, dtype: str
  col  v
0   A  1
1   B  2
2   C  3


In [31]:
# query / eval for readable filtering and fast expressions
dfq = pd.DataFrame({'a':[1,2,3], 'b':[10,20,30]})
print(dfq.query('a > 1'))
print(pd.eval('dfq.a + dfq.b'))
# sample / head / tail
print('sample:', dfq.sample(2))
# isin
print(dfq[dfq['a'].isin([1,3])])


   a   b
1  2  20
2  3  30
0    11
1    22
2    33
dtype: int64
sample:    a   b
1  2  20
2  3  30
   a   b
0  1  10
2  3  30


In [32]:
# explode (useful when a column contains lists)
dfe = pd.DataFrame({'id':[1,2], 'tags':[['x','y'], ['z']]})
print('explode ->', dfe.explode('tags'))
# get_dummies (one-hot encoding)
print(pd.get_dummies(pd.Series(['a','b','a']), prefix='cat'))
# cut / qcut for binning
nums = pd.Series([0,5,10,15,20])
print(pd.cut(nums, bins=[0,10,20]))
print(pd.qcut(nums, 2))


explode ->    id tags
0   1    x
0   1    y
1   2    z
   cat_a  cat_b
0   True  False
1  False   True
2   True  False
0             NaN
1     (0.0, 10.0]
2     (0.0, 10.0]
3    (10.0, 20.0]
4    (10.0, 20.0]
dtype: category
Categories (2, interval[int64, right]): [(0, 10] < (10, 20]]
0    (-0.001, 10.0]
1    (-0.001, 10.0]
2    (-0.001, 10.0]
3      (10.0, 20.0]
4      (10.0, 20.0]
dtype: category
Categories (2, interval[float64, right]): [(-0.001, 10.0] < (10.0, 20.0]]


In [33]:
# pct_change / shift / rank / corr / cov
dfp = pd.DataFrame({'val':[100,110,105,115]})
print('pct_change:', dfp['val'].pct_change())
print('shift(1):', dfp['val'].shift(1))
print('rank:', dfp['val'].rank())
print('corr matrix:', dfp.corr())


pct_change: 0         NaN
1    0.100000
2   -0.045455
3    0.095238
Name: val, dtype: float64
shift(1): 0      NaN
1    100.0
2    110.0
3    105.0
Name: val, dtype: float64
rank: 0    1.0
1    3.0
2    2.0
3    4.0
Name: val, dtype: float64
corr matrix:      val
val  1.0


In [34]:
# pipe for chaining transformations (readable pipelines)
def add_col(d):
    return d.assign(total=d.sum(axis=1))
piped = pd.DataFrame({'a':[1,2], 'b':[3,4]}).pipe(add_col)
print('pipe result:', piped)
# explode + groupby example: count tags per id
print(dfe.explode('tags').groupby('tags').size())


pipe result:    a  b  total
0  1  3      4
1  2  4      6
tags
x    1
y    1
z    1
dtype: int64


In [35]:
# performance helpers: memory_usage, astype optimizations
dfl = pd.DataFrame({'s': ['a']*1000 + ['b']*1000, 'n': list(range(2000))})
print('memory usage (bytes):', dfl.memory_usage(deep=True).sum())
dfl['s'] = dfl['s'].astype('category')
print('after category:', dfl.memory_usage(deep=True).sum())


memory usage (bytes): 34132
after category: 18151


## Common interview exercises (with hint code)

1. Given a DataFrame of user events, find the top-3 items per user (hint: groupby + apply or using `nlargest`).
2. Pivot hourly data into daily aggregates without explicit loops (hint: `resample`).
3. Given sales data, compute month-over-month growth per product (hint: groupby + pct_change).
4. Remove duplicate rows keeping the latest by timestamp (hint: sort_values + drop_duplicates(subset=..., keep='last')).

In [36]:
# Example: top-2 per group using groupby + nlargest
df_ex = pd.DataFrame({'user':[1,1,1,2,2], 'item':[10,11,12,10,13], 'score':[5,3,8,7,2]})
top2 = df_ex.groupby('user', group_keys=False).apply(lambda d: d.nlargest(2, 'score'))
print(top2)


   item  score
2    12      8
0    10      5
3    10      7
4    13      2


## Gotchas and best practices

- Avoid Python loops over rows; prefer vectorized operations or `apply` only when necessary.
- Be explicit with dtypes when loading large CSVs to reduce memory usage.
- Use `inplace=False` patterns (assignment) for clearer code; `inplace=True` is rarely faster.
- When performance matters, consider using `pd.eval`, `numexpr`, or switching to Dask for out-of-core data.

---
**Next steps:** Run example cells in your environment to verify outputs and I can add additional interview-style exercises (NLP/time-series/joins optimizations) on request.